# SciFact — Recuperación de Información

**Corpus:** SciFact (BEIR). **Unidad:** documento completo; IDs originales.

Se utiliza el mismo motor que la API. La primera ejecución descarga el corpus BEIR SciFact (incluye documentos, consultas y qrels).

**Nombre**: David Pilataxi
**Fecha**: 18 de noviembre de 2026


**Examen de Primer Bimestre**

In [17]:
from engine import SearchEngine, load_test, evaluate, tokenize
engine = SearchEngine()
print("Documentos:",len(engine.docs))
print("Vocabulario:",len(engine.vocab))
print("Términos en el índice invertido:",len(engine.inverted))

Documentos: 5183
Vocabulario: 20758
Términos en el índice invertido: 20758


## Preprocesamiento e indexación

Se unen título y texto de cada documento (sin fragmentarlo), se convierten a minúsculas, se tokeniza con expresión regular para palabras inglesas, se eliminan stopwords inglesas y se aplica Porter stemming. Se crea un índice invertido de términos a documentos, y una matriz dispersa de frecuencias. Se preserva `_id`. TF-IDF usa IDF suavizado y normalización L2; BM25 usa k1=1.5, b=0.75 e IDF positiva.

In [18]:
ejemplo = "Scientific studies show the effects of inflammation"
print("Ejemplo preprocesado:", tokenize(ejemplo))
print("Primer ID original:",engine.ids[0])
print("Primeros términos del vocabulario:",list(engine.vocab)[:20])

Ejemplo preprocesado: ['scientif', 'studi', 'effect', 'inflamm']
Primer ID original: 4983
Primeros términos del vocabulario: ['microstructur', 'develop', 'human', 'newborn', 'cerebr', 'white', 'matter', 'assess', 'vivo', 'diffus', 'tensor', 'magnet', 'reson', 'imag', 'alter', 'architectur', 'brain', 'affect', 'cortic', 'result']


## Pruebas de búsqueda

In [19]:
consulta = "Does vitamin D reduce the risk of respiratory infections?"
for modelo in ["tfidf", "bm25"]:
    print("\n",modelo.upper())
    for item in engine.search(consulta, modelo, 5):
        print(item["rank"],item["document_id"],round(item["score"],4),item["title"][:70])


 TFIDF
1 23267371 0.4283 Vitamin D: The "sunshine" vitamin.
2 12009265 0.3959 Vitamins E and C in the prevention of prostate and total cancer in men
3 21553394 0.3472 Vitamin D and obesity: current perspectives and future directions.
4 30720103 0.3346 Vitamin D status: measurement, interpretation, and clinical applicatio
5 21380348 0.3128 Intracellular transport of fat-soluble vitamins A and E.

 BM25
1 23267371 15.7574 Vitamin D: The "sunshine" vitamin.
2 12009265 15.6779 Vitamins E and C in the prevention of prostate and total cancer in men
3 38551172 15.2019 Mammographic density, plasma vitamin D levels and risk of breast cance
4 12810152 14.5865 Folate and vitamin B6 from diet and supplements in relation to risk of
5 30720103 14.1456 Vitamin D status: measurement, interpretation, and clinical applicatio


## Pruebas de consistencia
Esta celda comprueba límites, ranking descendente, unicidad y pertenencia de IDs, y consultas sin términos indexados.


In [20]:
# Comprobaciones de orden, IDs originales y límites para ambos modelos
for modelo in ("bm25", "tfidf"):
    encontrados = engine.search("respiratory infection", modelo, top_k=5)
    assert len(encontrados) <= 5
    assert len({r["document_id"] for r in encontrados}) == len(encontrados)
    assert all(r["document_id"] in engine.ids for r in encontrados)
    assert all(a["score"] >= b["score"] for a, b in zip(encontrados, encontrados[1:]))
    assert engine.search("zzzxxyyqqq", modelo) == []
print("Verificación básica de rankings: OK")


Verificación básica de rankings: OK


## Evaluación sobre TODAS las consultas test con qrels

Precision@10 = relevantes recuperados / 10. Recall@10 = relevantes recuperados / total relevantes de la consulta. NDCG@10 = DCG@10 / IDCG@10, utilizando las calificaciones originales de qrels. Se promedian las métricas por consulta.

In [21]:
queries,qrels=load_test()
print("Consultas test con qrels:",len(qrels))
print("Ejemplo de consultas evaluadas:")
for qid in list(qrels)[:10]: print(qid,queries.get(qid,""))

Consultas test con qrels: 300
Ejemplo de consultas evaluadas:
1 0-dimensional biomaterials show inductive properties.
3 1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.
5 1/2000 in UK have abnormal PrP positivity.
13 5% of perinatal mortality is due to low birth weight.
36 A deficiency of vitamin B12 increases blood levels of homocysteine.
42 A high microerythrocyte count raises vulnerability to severe anemia in homozygous alpha (+)- thalassemia trait subjects.
48 A total of 1,000 people in the UK are asymptomatic carriers of vCJD infection.
49 ADAR1 binds to Dicer to cleave pre-miRNA.
50 AIRE is expressed in some skin tumors.
51 ALDH1 expression is associated with better breast cancer outcomes.


In [22]:
tfidf_metrics,tfidf_rows=evaluate(engine,"tfidf")
bm25_metrics,bm25_rows=evaluate(engine,"bm25")
print("TF-IDF:",tfidf_metrics)
print("BM25:",bm25_metrics)

TF-IDF: {'precision@10': 0.08266666666666667, 'recall@10': 0.7558888888888889, 'ndcg@10': 0.5978292692157604}
BM25: {'precision@10': 0.09233333333333334, 'recall@10': 0.8386666666666667, 'ndcg@10': 0.6918501040101689}


In [23]:
import pandas as pd
comparison=pd.DataFrame([{"modelo":"TF-IDF",**tfidf_metrics},{"modelo":"BM25",**bm25_metrics}]).set_index("modelo")
display(comparison)
display(pd.DataFrame(bm25_rows).head(10))

,precision@10,recall@10,ndcg@10
modelo,,,
TF-IDF,0.082667,0.755889,0.597829
BM25,0.092333,0.838667,0.691850


,query_id,query,precision@10,recall@10,ndcg@10
0,1,0-dimensional biomaterials show inductive prop...,0.0,0.0,0.000000
1,3,"1,000 genomes project enables mapping of genet...",0.1,1.0,0.430677
2,5,1/2000 in UK have abnormal PrP positivity.,0.1,1.0,1.000000
3,13,5% of perinatal mortality is due to low birth ...,0.0,0.0,0.000000
4,36,A deficiency of vitamin B12 increases blood le...,0.1,0.5,0.264068
5,42,A high microerythrocyte count raises vulnerabi...,0.1,1.0,0.630930
6,48,"A total of 1,000 people in the UK are asymptom...",0.1,1.0,0.630930
7,49,ADAR1 binds to Dicer to cleave pre-miRNA.,0.1,1.0,1.000000
8,50,AIRE is expressed in some skin tumors.,0.1,1.0,1.000000
9,51,ALDH1 expression is associated with better bre...,0.1,1.0,1.000000


## Interpretación de los resultados obtenidos

Se evaluaron **300 consultas de la partición test** de SciFact con sus **qrels**, recuperando hasta **10 documentos** por consulta y calculando el **promedio macro** (cada consulta pesa lo mismo). Los documentos relevantes se definen como aquellos con calificación mayor que cero; para NDCG se utilizan las calificaciones originales.

| Métrica | TF-IDF (coseno) | BM25 | Diferencia BM25 − TF-IDF |
|:--|--:|--:|--:|
| Precision@10 | 0.0827 | **0.0923** | +0.0097 |
| Recall@10 | 0.7559 | **0.8387** | +0.0828 |
| NDCG@10 | 0.5978 | **0.6919** | +0.0940 |

**Análisis:** BM25 superó a TF-IDF en las tres métricas del conjunto evaluado. En promedio, de las diez posiciones de cada ranking, aproximadamente **0.923 documentos** fueron relevantes con BM25, frente a **0.827** con TF-IDF (Precision@10 × 10). Aunque esas precisiones parecen bajas, SciFact tiene relativamente pocos documentos juzgados relevantes por consulta: incluir siempre diez resultados penaliza la precisión incluso si se encuentran todos los relevantes. Esta observación no significa que todo resultado no marcado sea necesariamente irrelevante en sentido absoluto; las métricas utilizan los qrels disponibles como referencia.

El **Recall@10** de BM25 indica que recuperó aproximadamente **83.87 %** de los documentos etiquetados como relevantes dentro de las diez primeras posiciones, frente al **75.59 %** de TF-IDF. El mayor **NDCG@10** indica además un mejor ordenamiento conforme a los grados de relevancia de los qrels. La diferencia observada es descriptiva: sin intervalos de confianza ni pruebas estadísticas no debe afirmarse que sea estadísticamente significativa ni que BM25 siempre gane ante cualquier consulta.

**Explicación técnica:** ambos son métodos léxicos y dependen de coincidencias entre términos. En esta implementación, BM25 incluye saturación de frecuencia y normalización de longitud (k1 = 1.5, b = 0.75), mientras que TF-IDF utiliza IDF suavizado y coseno con normalización L2; esas diferencias pueden explicar parte de los rankings distintos. Las escalas de *score* de ambos modelos son diferentes y **no se comparan numéricamente entre modelos**.

**Limitaciones:** el preprocesamiento empleado elimina números, signos y símbolos, lo que puede perder información importante en consultas científicas (por ejemplo, cantidades o identificadores); asimismo, stemming y stopwords pueden afectar términos técnicos. El corpus se trata como documentos completos, sin fragmentación. Los resultados y conclusiones de esta celda corresponden exclusivamente a esta configuración y a las 300 consultas test evaluadas.


## API pública

La API usa exactamente el mismo motor: `GET /health` y `POST /search`. Consulte README.md para publicarla en Render y probarla en Postman.